In [ ]:
import requests
import pandas as pd
def get_10_day_weather_summary(lat, lon,days):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&daily=temperature_2m_mean,relative_humidity_2m_mean,precipitation_sum&forecast_days={days}&timezone=auto"
    response = requests.get(url)
    
    if response.status_code != 200:
        raise Exception(f"Failed to fetch weather data: {response.text}")
    # print(response.json())
    data = response.json()["daily"]
    daily_df = pd.DataFrame({
        "Date": data["time"],
        "Avg_Temp_C": data["temperature_2m_mean"],
        "Avg_Humidity_%": data["relative_humidity_2m_mean"],
        "Rainfall_mm": data["precipitation_sum"]
    })
    
    weather_summary = {
        "temperature": round(daily_df["Avg_Temp_C"].mean(), 2),
        "humidity": round(daily_df["Avg_Humidity_%"].mean(), 2),
        "rainfall": round(daily_df["Rainfall_mm"].sum(), 2)  
    }
    
    return daily_df, weather_summary

latitude = 25.4358
longitude = 81.8463

daily_breakdown, ml_features = get_10_day_weather_summary(latitude, longitude , 16)

print("--- 10-Day Daily Forecast ---")
print(daily_breakdown)

print("\n--- Aggregated Features for Crop Prediction Model ---")
print(ml_features)

--- 10-Day Daily Forecast ---
          Date  Avg_Temp_C  Avg_Humidity_%  Rainfall_mm
0   2026-08-19        27.4              91         51.7
1   2026-08-20        28.0              89          8.9
2   2026-08-21        28.3              89          7.4
3   2026-08-22        27.8              90          8.5
4   2026-08-23        28.1              87          4.0
5   2026-08-24        28.6              87          8.7
6   2026-08-25        27.9              89         14.7
7   2026-08-26        28.5              88         13.2
8   2026-08-27        28.5              91         11.4
9   2026-08-28        28.0              89         12.0
10  2026-08-29        26.7              94         46.2
11  2026-08-30        27.6              88          4.8
12  2026-08-31        27.8              86          5.4
13  2026-09-01        28.1              79          2.4
14  2026-09-02        27.4              90         14.4
15  2026-09-03        29.5              78          8.7

--- Aggregated Fe

In [3]:
import requests
def get_live_location_ip():
    try:
        response = requests.get("https://ipapi.co/json/", timeout=5)
        data = response.json()
        latitude = data.get("latitude")
        longitude = data.get("longitude")
        city = data.get("city")
        region = data.get("region")
        
        return latitude, longitude, f"{city}, {region}"
    except Exception as e:
        print(f"Error fetching IP location: {e}")
        return None, None, None

# Example usage
lat, lon, location_name = get_live_location_ip()
print(f"Location: {location_name}")
print(f"Latitude: {lat}, Longitude: {lon}")

Location: None, None
Latitude: None, Longitude: None


In [ ]:
from plyer import gps

def on_location(**kwargs):
    lat = kwargs.get('lat')
    lon = kwargs.get('lon')
    print(f"GPS Coordinates: Latitude = {lat}, Longitude = {lon}")
    gps.stop()

def get_mobile_gps():
    try:
        gps.configure(on_location=on_location)
        gps.start(minTime=1000, minDistance=1) # Update every 1 sec or 1 meter
    except NotImplementedError:
        print("GPS hardware API is not supported on this operating system.")

get_mobile_gps()

GPS hardware API is not supported on this operating system.


In [ ]:
import http.server
import socketserver
import webbrowser
import json
import threading
import requests 

location_result = {}

class GPSHandler(http.server.SimpleHTTPRequestHandler):
    def do_GET(self):
        if self.path == '/':
            self.send_response(200)
            self.send_header('Content-type', 'text/html')
            self.end_headers()
            
            html = """
            <!DOCTYPE html>
            <html>
            <head><title>GPS Capture</title></head>
            <body style="font-family: Arial, sans-serif; text-align: center; margin-top: 50px;">
                <h2 id="status">Requesting Hardware GPS Location...</h2>
                <p>Please click <b>"Allow"</b> when prompted by your browser.</p>
                
                <script>
                    if ("geolocation" in navigator) {
                        navigator.geolocation.getCurrentPosition(
                            function(position) {
                                document.getElementById("status").innerText = "GPS Coordinates Captured! You may close this tab.";
                                fetch('/submit', {
                                    method: 'POST',
                                    headers: {'Content-Type': 'application/json'},
                                    body: JSON.stringify({
                                        lat: position.coords.latitude,
                                        lon: position.coords.longitude,
                                        accuracy: position.coords.accuracy
                                    })
                                });
                            },
                            function(error) {
                                document.getElementById("status").innerText = "Error fetching GPS: " + error.message;
                            },
                            { enableHighAccuracy: true, timeout: 15000, maximumAge: 0 }
                        );
                    } else {
                        document.getElementById("status").innerText = "GPS is not supported by this browser.";
                    }
                </script>
            </body>
            </html>
            """
            self.wfile.write(html.encode('utf-8'))
        else:
            self.send_error(404)

    def do_POST(self):
        if self.path == '/submit':
            content_length = int(self.headers['Content-Length'])
            post_data = self.rfile.read(content_length)
            
            global location_result
            location_result = json.loads(post_data.decode('utf-8'))
            
            self.send_response(200)
            self.end_headers()
            
            threading.Thread(target=self.server.shutdown).start()

    def log_message(self, format, *args):
        return

def get_live_gps_location(port=8999):
    """Launches local server to fetch browser hardware GPS coordinates."""
    with socketserver.TCPServer(("127.0.0.1", port), GPSHandler) as httpd:
        print("Opening browser to request hardware GPS permissions...")
        webbrowser.open(f"http://127.0.0.1:{port}/")
        httpd.serve_forever()
    
    lat = location_result.get('lat')
    lon = location_result.get('lon')
    acc = location_result.get('accuracy')
    
    return lat, lon, acc

def reverse_geocode(lat, lon):
    """Converts Latitude & Longitude into City, State, Country, and Postal Code."""
    url = f"https://nominatim.openstreetmap.org/reverse?lat={lat}&lon={lon}&format=json"
    headers = {"User-Agent": "AgriCropApp/1.0"} 
    
    try:
        response = requests.get(url, headers=headers, timeout=5)
        if response.status_code == 200:
            address = response.json().get("address", {})
            print(response.json())
        
            city = address.get("city") or address.get("town") or address.get("village") or address.get("county") or "Unknown"
            state = address.get("state", "Unknown")
            country = address.get("country", "Unknown")
            pincode = address.get("postcode", "Unknown")
            suburb = address.get("suburb") or address.get("neighbourhood") or ""
            
            return {
                "city": city,
                "state": state,
                "country": country,
                "pincode": pincode,
                "area": suburb,
                "full_address": response.json().get("display_name", "Unknown")
            }
    except Exception as e:
        print(f"Error fetching address details: {e}")
        
    return None
def get_lat_long():
    """Main wrapper function to fetch GPS coordinates and display report."""
    latitude, longitude, accuracy = get_live_gps_location()
    
    if latitude and longitude:
        address_info = reverse_geocode(latitude, longitude)
        
        print("\n====================================")
        print("     GPS LOCATION DETAILED REPORT    ")
        print("====================================")
        print(f"Latitude     : {latitude}")
        print(f"Longitude    : {longitude}")
        print(f"Accuracy     : ±{round(accuracy, 2)} meters")
        print("------------------------------------")
        if address_info:
            print(f"Area / Suburb: {address_info['area']}")
            print(f"City / Town  : {address_info['city']}")
            print(f"State        : {address_info['state']}")
            print(f"Pincode      : {address_info['pincode']}")
            print(f"Country      : {address_info['country']}")
            print("------------------------------------")
            print(f"Full Address : {address_info['full_address']}")
        print("====================================")
        return latitude, longitude
    else:
        print("\nFailed to capture GPS location.")
        return None, None

In [6]:
if __name__ == "__main__":
    lati, long = get_lat_long()
    print(f"Captured Coordinates: {lati}, {long}")

Opening browser to request hardware GPS permissions...
{'place_id': 243170814, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'way', 'osm_id': 956028475, 'lat': '25.4930703', 'lon': '81.8678152', 'class': 'highway', 'type': 'residential', 'place_rank': 26, 'importance': 0.05340043869687949, 'addresstype': 'road', 'name': '', 'display_name': 'Teliyarganj, Prayagraj, Allahabad, Prayagraj, Uttar Pradesh, 211001, India', 'address': {'suburb': 'Teliyarganj', 'city': 'Prayagraj', 'county': 'Allahabad', 'state_district': 'Prayagraj', 'state': 'Uttar Pradesh', 'ISO3166-2-lvl4': 'IN-UP', 'postcode': '211001', 'country': 'India', 'country_code': 'in'}, 'boundingbox': ['25.4928198', '25.4947381', '81.8676093', '81.8692443']}

     GPS LOCATION DETAILED REPORT    
Latitude     : 25.493309620586587
Longitude    : 81.8675241201712
Accuracy     : ±108 meters
------------------------------------
Area / Suburb: Teliyarganj
City / Town  : Prayagraj
State 

In [32]:
import joblib
loaded_model = joblib.load("xgb_crop_model.joblib")
loaded_label_encoder = joblib.load("label_encoder.joblib")

c:\Users\KAILASH\anaconda3\envs\virtual\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [34]:
import pandas as pd
import numpy as np

def test(lat, lon):
    if lat is None or lon is None:
        raise ValueError("Latitude and Longitude cannot be None. Please enable GPS permissions and try again.")
    daily_breakdown, ml_features = get_10_day_weather_summary(lat, lon, days=16)
    N = float(input("Enter Nitrogen (N): "))
    P = float(input("Enter Phosphorus (P): "))
    K = float(input("Enter Potassium (K): "))
    ph = float(input("Enter pH (e.g. 6.5): "))
    ml_features['N'] = N
    ml_features['P'] = P
    ml_features['K'] = K
    ml_features['ph'] = ph
    expected_feature_order = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
    sample = pd.DataFrame([ml_features])[expected_feature_order]
    probs = loaded_model.predict_proba(sample)[0]
    top_5_indices = np.argsort(probs)[-5:][::-1]
    top_5_crops = loaded_label_encoder.inverse_transform(top_5_indices)
    top_5_scores = probs[top_5_indices] * 100
    return pd.DataFrame({
        "Rank": [f"#{i+1}" for i in range(5)],
        "Crop": top_5_crops,
        "Confidence": [f"{score:.2f}%" for score in top_5_scores]
    })

lat, long = get_lat_long()
if lat is not None and long is not None:
    top_crops_df = test(lat, long)
    print(top_crops_df)
else:
    print("Execution stopped: Invalid GPS coordinates.")

Opening browser to request hardware GPS permissions...


{'place_id': 243440386, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'node', 'osm_id': 12538749804, 'lat': '25.4927832', 'lon': '81.8640478', 'class': 'amenity', 'type': 'library', 'place_rank': 30, 'importance': 6.710536354615118e-05, 'addresstype': 'amenity', 'name': 'Central Library', 'display_name': 'Central Library, UNDERPASS, Hostel area, MNNIT, Teliyarganj, Prayagraj, Allahabad, Prayagraj, Uttar Pradesh, 211001, India', 'address': {'amenity': 'Central Library', 'road': 'UNDERPASS', 'residential': 'Hostel area, MNNIT', 'suburb': 'Teliyarganj', 'city': 'Prayagraj', 'county': 'Allahabad', 'state_district': 'Prayagraj', 'state': 'Uttar Pradesh', 'ISO3166-2-lvl4': 'IN-UP', 'postcode': '211001', 'country': 'India', 'country_code': 'in'}, 'boundingbox': ['25.4927332', '25.4928332', '81.8639978', '81.8640978']}

     GPS LOCATION DETAILED REPORT    
Latitude     : 25.492782653820452
Longitude    : 81.86409876703051
Accuracy     : ±108 m

In [ ]:
import pandas as pd

def get_important_crop_data(top_crops_df, csv_path='combined_data.csv'):
    crop_list = top_crops_df['Crop'].tolist()
    
    df = pd.read_csv(csv_path)
    
    # Filter the dataset for only the predicted crops
    filtered_data = df[df['crop_slug'].isin(crop_list)].copy()
    
    # Define which columns are considered "important"
    important_columns = [
        'crop_name', 'mandi_name', 'district_state', 
        'min_price', 'max_price', 'modal_price', 
        'season', 'sowing_window'
    ]
    
    # Keep only columns that exist in the csv
    columns_to_keep = [col for col in important_columns if col in filtered_data.columns]
    
    return filtered_data[columns_to_keep]

# Use the lat and long variables that were already saved in the previous cells
if 'lat' in locals() and 'long' in locals() and lat is not None and long is not None:
    print("--- 1. Predicting Top Crops ---")
    predicted_crops = test(lat, long)
    display(predicted_crops)
    
    print("\n--- 2. Fetching Market Data for Predicted Crops ---")
    # Pass the output of your test() function directly into the new function
    market_data = get_important_crop_data(predicted_crops)
    display(market_data)
else:
    print("Error: 'lat' and 'long' are not defined. Please run the GPS location cell first.")


--- 1. Predicting Top Crops ---


,Rank,Crop,Confidence
0,#1,cauliflower,16.07%
1,#2,banana,13.25%
2,#3,blackgram,5.36%
3,#4,cardamom,3.57%
4,#5,cabbage,3.56%



--- 2. Fetching Market Data for Predicted Crops ---


,crop_name,mandi_name,district_state,min_price,max_price,modal_price,season,sowing_window
584,ð¥¬ Cauliflower,Adilabad(Rythu Bazar) APMC,"Adilabad, Telangana","â¹9,000","â¹9,200","â¹9,000",Rabi Â,Sep-Nov
585,ð¥¬ Cauliflower,Balugaon APMC,"Khurda, Odisha","â¹6,500","â¹6,500","â¹6,500",Rabi Â,Sep-Nov
586,ð¥¬ Cauliflower,Baghmara APMC,"South Garo Hills, Meghalaya","â¹5,000","â¹6,000","â¹5,500",Rabi Â,Sep-Nov
587,ð¥¬ Cauliflower,Myladi(Uzhavar Sandhai ),"Nagercoil (Kannyiakumari), Tamil Nadu","â¹5,000","â¹5,500","â¹5,250",Rabi Â,Sep-Nov
588,ð¥¬ Cauliflower,Ramanathapuram(Uzhavar Sandhai ),"Ramanathapuram, Tamil Nadu","â¹4,500","â¹5,500","â¹5,000",Rabi Â,Sep-Nov
...,...,...,...,...,...,...,...,...
981,ð Banana,Peravurani(Uzhavar Sandhai),"Thanjavur, Tamil Nadu","â¹3,000","â¹8,000","â¹4,200",Season,NaN
982,ð Banana,PMY Kather Solan,"Solan, Himachal Pradesh","â¹2,400","â¹5,400","â¹4,200",Season,NaN
983,ð Banana,Mecheri(Uzhavar Sandhai),"Salem, Tamil Nadu","â¹3,000","â¹6,000","â¹4,139",Season,NaN
984,ð Banana,Kallachi Market,"Kozhikode(Calicut), Keralam","â¹3,000","â¹5,000","â¹4,133",Season,NaN
